In [61]:
!pip install -q chromadb google-genai streamlit python-dotenv langchain-community langchain-textsplitters

!npm install -g localtunnel

ERROR: Could not find a version that satisfies the requirement langchain-textsplitters (from versions: none)
ERROR: No matching distribution found for langchain-textsplitters
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

## Task 11.1 & 11.2: Embeddings & Similarity Calculations

In [62]:
import os
import numpy as np
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

def get_embedding(text: str):
    """
    Generate text embeddings using standard model routing.
    Tries text-embedding-004, falls back to gemini-embedding-2 if project-restricted.
    """
    try:
        response = client.models.embed_content(
            model="text-embedding-004",
            contents=text
        )
        return response.embeddings[0].values
    except Exception:
        response = client.models.embed_content(
            model="gemini-embedding-2",
            contents=text
        )
        return response.embeddings[0].values

def cosine_similarity(vec1, vec2):
    """Calculate the cosine similarity between two numeric vectors."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

test_embed = get_embedding("vacation policy")
print(f" Success! Gemini Embedding Vector Length: {len(test_embed)}")

phrases = ['vacation policy', 'time off rules', 'PTO guidelines', 'dress code requirements']
embeddings = [get_embedding(p) for p in phrases]

print(f'\nComparing "{phrases[0]}" with alternative phrases:')
for i, phrase in enumerate(phrases[1:], start=1):
    similarity = cosine_similarity(embeddings[0], embeddings[i])
    print(f"{phrase:30} Similarity Score: {similarity:.4f}")

 Success! Gemini Embedding Vector Length: 3072

Comparing "vacation policy" with alternative phrases:
time off rules                 Similarity Score: 0.8531
PTO guidelines                 Similarity Score: 0.7201
dress code requirements        Similarity Score: 0.6544


## Lab 11 - Task 2.1 & 2.2: Set Up and Index ChromaDB

In [63]:

import os
import chromadb
from google.genai import types

os.makedirs("company_docs", exist_ok=True)
policies = {
    "vacation.txt": "Employees receive 20 days of paid time off (PTO) annually. Vacation requests must be submitted 2 weeks in advance.",
    "remote_work.txt": "Our WFH guidelines allow team members to work remotely up to 3 days a week. Core collaborating hours are 10 AM to 4 PM.",
    "parental_leave.txt": "Maternity leave offers 16 weeks of fully paid leave for new mothers. Paternal leave offers 4 weeks of paid time off."
}

for filename, content in policies.items():
    with open(f"company_docs/{filename}", "w") as f:
        f.write(content)

class GeminiEmbeddingFunction(chromadb.EmbeddingFunction):
    def __call__(self, input: chromadb.Documents) -> chromadb.Embeddings:
        vectors = []
        for text in input:
            try:
                response = client.models.embed_content(
                    model="text-embedding-004",
                    contents=text,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                vectors.append(response.embeddings[0].values)
            except Exception:
                response = client.models.embed_content(
                    model="gemini-embedding-2",
                    contents=text,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                vectors.append(response.embeddings[0].values)
        return vectors

chroma_client = chromadb.PersistentClient(path='./chroma_db')
gemini_ef = GeminiEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="company_docs",
    embedding_function=gemini_ef,
    metadata={"description": "Company policy documents using Gemini embeddings"}
)

def split_text_native(text, chunk_size=500, chunk_overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - chunk_overlap)
    return chunks

all_chunks = []
chunk_metadata = []

for filename in os.listdir("company_docs"):
    if filename.endswith(".txt"):
        file_path = os.path.join("company_docs", filename)
        with open(file_path, "r") as f:
            file_content = f.read()
            file_chunks = split_text_native(file_content, chunk_size=500, chunk_overlap=50)

            for chunk in file_chunks:
                all_chunks.append(chunk)
                chunk_metadata.append({"source": f"company_docs/{filename}"})

if collection.count() == 0:
    collection.add(
        documents=all_chunks,
        ids=[f"doc_{i}" for i in range(len(all_chunks))],
        metadatas=chunk_metadata
    )
    print(f" Success! Added {collection.count()} chunks to ChromaDB.")
else:
    print(f" Collection already populated. Count: {collection.count()}")

 Collection already populated. Count: 3


/tmp/ipykernel_2948/23730066.py:37: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  gemini_ef = GeminiEmbeddingFunction()


## Task 3.1: Test Vector Search with Synonyms

In [64]:

def vector_search(query: str, n_results: int = 2):
    """
    Search the ChromaDB vector database using semantic similarity.
    Returns the raw search results dictionary from ChromaDB.
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

test_queries = [
    'time off policy',
    'WFH guidelines',
    'maternity leave'
]

print("--- Running Semantic Vector Search Tests ---")
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")

    results = vector_search(query, n_results=1)

    if results['documents'] and len(results['documents'][0]) > 0:
        matched_doc = results['documents'][0][0]
        matched_dist = results['distances'][0][0] if 'distances' in results and results['distances'] else 0.0
        matched_meta = results['metadatas'][0][0] if 'metadatas' in results and results['metadatas'] else {}

        print(f"Matched Source : {matched_meta.get('source', 'unknown')}")
        print(f"Distance Score : {matched_dist:.4f}")
        print(f"Snippet        : {matched_doc[:150]}...")
    else:
        print(" No relevant document chunks found.")

--- Running Semantic Vector Search Tests ---

Query: 'time off policy'
Matched Source : company_docs/vacation.txt
Distance Score : 0.4791
Snippet        : Employees receive 20 days of paid time off (PTO) annually. Vacation requests must be submitted 2 weeks in advance....

Query: 'WFH guidelines'
Matched Source : company_docs/remote_work.txt
Distance Score : 0.4246
Snippet        : Our WFH guidelines allow team members to work remotely up to 3 days a week. Core collaborating hours are 10 AM to 4 PM....

Query: 'maternity leave'
Matched Source : company_docs/parental_leave.txt
Distance Score : 0.4971
Snippet        : Maternity leave offers 16 weeks of fully paid leave for new mothers. Paternal leave offers 4 weeks of paid time off....


##Task 3.2: Build Semantic RAG Pipeline

In [65]:

from google.genai import types

def semantic_rag(query: str, n_results: int = 2):
    """
    Complete semantic RAG pipeline:
    1. Vector search to pull context from ChromaDB.
    2. Format prompt with strict context boundaries.
    3. Generate clean response using Gemini 2.5 Flash.
    """
    results = vector_search(query, n_results=n_results)

    if not results['documents'] or len(results['documents'][0]) == 0:
        return 'No relevant information found.'

    context = "\n\n---\n\n".join(results['documents'][0])

    system_instruction = (
        "You are a helpful HR assistant. Answer using ONLY the context provided below. "
        "If the information is not in the context, explicitly say so. Be concise and friendly."
    )
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0
        )
    )
    return response.text

questions = [
    'How much time off do employees get?',
    'Can I work from home?',
    'What is the maternity leave policy?'
]

print("--- Testing Semantic RAG Model Responses ---")
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {semantic_rag(q)}")

--- Testing Semantic RAG Model Responses ---

Q: How much time off do employees get?
A: Employees receive 20 days of paid time off (PTO) annually.

Q: Can I work from home?
A: Yes, our WFH guidelines allow team members to work remotely up to 3 days a week.

Q: What is the maternity leave policy?
A: Maternity leave offers 16 weeks of fully paid leave for new mothers.


##Lab 11 - Bonus: Keyword vs Semantic Comparison

In [66]:

raw_chunks = [
    {"text": "Employees receive 20 days of paid time off (PTO) annually. Vacation requests must be submitted 2 weeks in advance.", "source": "vacation.txt"},
    {"text": "Our WFH guidelines allow team members to work remotely up to 3 days a week. Core collaborating hours are 10 AM to 4 PM.", "source": "remote_work.txt"},
    {"text": "Maternity leave offers 16 weeks of fully paid leave for new mothers. Paternal leave offers 4 weeks of paid time off.", "source": "parental_leave.txt"}
]

def keyword_search(query, top_k=2):
    scored = []
    query_words = query.lower().split()
    for chunk in raw_chunks:
        score = sum(chunk["text"].lower().count(word) for word in query_words)
        if score > 0:
            scored.append((score, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [item[1] for item in scored[:top_k]]

synonym_query = "PTO policy"

print("---  KEYWORD SEARCH RESULTS ---")
kw_results = keyword_search(synonym_query)
print(f"Found: {len(kw_results)} results matching exact keywords.")
for doc in kw_results:
    print(f"-> Match from {doc['source']}: {doc['text'][:80]}...")

print("\n- SEMANTIC SEARCH RESULTS ---")
sem_results = vector_search(synonym_query, n_results=1)
if sem_results['documents'] and len(sem_results['documents'][0]) > 0:
    print(f"Found: {len(sem_results['documents'][0])} semantic match result(s).")
    print(f"Top Result: {sem_results['documents'][0][0][:120]}...")
else:
    print("Found: 0 semantic results.")

---  KEYWORD SEARCH RESULTS ---
Found: 1 results matching exact keywords.
-> Match from vacation.txt: Employees receive 20 days of paid time off (PTO) annually. Vacation requests mus...

- SEMANTIC SEARCH RESULTS ---
Found: 1 semantic match result(s).
Top Result: Employees receive 20 days of paid time off (PTO) annually. Vacation requests must be submitted 2 weeks in advance....


## Lab 12 - Step 1: Write the Streamlit Application Code

In [73]:
%%writefile app.py

import os
import streamlit as st
import chromadb
from google import genai
from google.genai import types

st.set_page_config(page_title='AI HR Assistant', layout='wide', page_icon='🏢')

try:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        st.error(" Error: GEMINI_API_KEY environment variable is missing from the application context.")
        st.stop()

    client = genai.Client(api_key=GEMINI_API_KEY)

    class StreamlitGeminiEmbedder(chromadb.EmbeddingFunction):
        def __call__(self, input: chromadb.Documents) -> chromadb.Embeddings:
            vectors = []
            for text in input:
                try:
                    response = client.models.embed_content(
                        model="text-embedding-004", contents=text,
                        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                    )
                    vectors.append(response.embeddings[0].values)
                except Exception:
                    response = client.models.embed_content(
                        model="gemini-embedding-2", contents=text,
                        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                    )
                    vectors.append(response.embeddings[0].values)
            return vectors

    @st.cache_resource
    def load_vector_db():
        db_client = chromadb.PersistentClient(path='./chroma_db')
        ef = StreamlitGeminiEmbedder()
        return db_client.get_collection(name='company_docs', embedding_function=ef)

    collection = load_vector_db()

except FileNotFoundError:
    st.error(' 11 X Error: ChromaDB local vector storage database directory path not found.')
    st.info('Please run your Lab 11 notebook cells first to index the mock company documents into ChromaDB.')
    st.stop()
except Exception as e:
    st.error(f' App Initialization Error: {str(e)}')
    st.stop()

st.title('🏢 Company Knowledge Assistant')
st.markdown("Query company leave schemas, remote working mandates, and HR policies instantly powered by a semantic RAG loop.")

with st.sidebar:
    st.header('System Status')
    st.markdown('A scalable production-grade assistant running on top of **Gemini 2.5 Flash** and **ChromaDB**.')
    st.divider()

    st.metric('Indexed Knowledge Vectors', collection.count())
    if 'messages' in st.session_state:
        st.metric('Conversation Length', len(st.session_state.messages))

    st.divider()
    if st.button('Clear Conversation History', use_container_width=True):
        st.session_state.messages = []
        st.rerun()

if 'messages' not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.write(msg['content'])

if len(st.session_state.messages) == 0:
    welcome_text = "Hello there! I am your AI corporate helper. Ask me questions like: 'How many days off do I get?' or 'What are the remote work rules?' to get started."
    with st.chat_message('assistant'):
        st.write(welcome_text)

if prompt := st.chat_input('Ask an internal policy question...'):
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('user'):
        st.write(prompt)

    with st.chat_message('assistant'):
        with st.spinner('Scanning internal company vector knowledge base...'):
            try:
                results = collection.query(query_texts=[prompt], n_results=2)

                if results['documents'] and len(results['documents'][0]) > 0:
                    context = "\n\n---\n\n".join(results['documents'][0])

                    system_instruction = "You are a helpful HR assistant. Answer using ONLY the context provided. If not in context, say so. Be concise and friendly."
                    prompt_payload = f"Context:\n{context}\n\nQuestion: {prompt}\nAnswer:"

                    response = client.models.generate_content(
                        model='gemini-2.5-flash',
                        contents=prompt_payload,
                        config=types.GenerateContentConfig(
                            system_instruction=system_instruction,
                            temperature=0.0
                        )
                    )
                    answer = response.text
                else:
                    answer = "No contextual background documentation records match those terms."
            except Exception as runtime_error:
                answer = f"Engine Execution Error: {str(runtime_error)}"

            st.write(answer)
            st.session_state.messages.append({'role': 'assistant', 'content': answer})

Overwriting app.py


##Lab 12 - Step 2: Inject Credentials and Initialize the Streamlit Server Background Process

In [74]:

import os
import subprocess
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

print("COPY THIS NUMERICAL TUNNEL PASSWORD IP:")
!curl ipv4.icanhazip.com
print("-" * 65)

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
print("Streamlit system engine actively spinning up on local port 8501...")

COPY THIS NUMERICAL TUNNEL PASSWORD IP:
136.117.51.217
-----------------------------------------------------------------
Streamlit system engine actively spinning up on local port 8501...


## Lab 12 - Step 3: Launch Localtunnel Route to Access the App

In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://gold-forks-pull.loca.lt
